In [1]:
from constants import users_list, data_path
from lib import spoti, genre_normalizer, plotting, preprocessing, dimensionality_reduction

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import json
import os
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import mean_squared_error
import random
import time
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors, NeighborhoodComponentsAnalysis
from sklearn.metrics import pairwise_distances
import pickle
import numpy as np
from typing import List, Dict, Tuple, Optional, Any, Callable
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from IPython.display import display, HTML
from implicit.als import AlternatingLeastSquares
from implicit.nearest_neighbours import bm25_weight, tfidf_weight
from implicit.lmf import LogisticMatrixFactorization
from scipy.sparse import csr_matrix

/Users/owen/local/unibe/ml/spotify-recommender/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Import data

In [2]:
df = spoti.load_all_tracks(
    base_path=data_path.DATA_PATH,
    users=users_list.USERS,
    load_spotify_tracks=False,
    penality_factors={"short_term": 1, "medium_term": 1, "long_term": 1},
)
df

,album,artists,available_markets,disc_number,duration_ms,explicit,external_ids,external_urls,href,id,...,time_range,affinity,username,release_year,normalized_genres,added_at,episode,track,added_by,playlist_id
0,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,196426,False,{'isrc': 'USSM12301260'},{'spotify': 'https://open.spotify.com/track/75...,https://api.spotify.com/v1/tracks/75rqqKvzJCGv...,75rqqKvzJCGv2oq9C4yFDt,...,medium_term,1.00,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
1,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,137533,True,{'isrc': 'USSM12109218'},{'spotify': 'https://open.spotify.com/track/2F...,https://api.spotify.com/v1/tracks/2FYGZDfsAnNs...,2FYGZDfsAnNsrm1gVbyKnG,...,medium_term,0.98,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
2,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,162906,True,{'isrc': 'USSM12109222'},{'spotify': 'https://open.spotify.com/track/4k...,https://api.spotify.com/v1/tracks/4kroNlz8BTfs...,4kroNlz8BTfswE4M0i3YCh,...,medium_term,0.96,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
3,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,89749,False,{'isrc': 'USSM12208854'},{'spotify': 'https://open.spotify.com/track/2N...,https://api.spotify.com/v1/tracks/2N3YZ075lq9z...,2N3YZ075lq9z1ObaAiX6l1,...,medium_term,0.94,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
4,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,174044,False,{'isrc': 'USSM12300114'},{'spotify': 'https://open.spotify.com/track/2S...,https://api.spotify.com/v1/tracks/2SiAcexM2p1y...,2SiAcexM2p1yX6joESbehd,...,medium_term,0.92,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12424,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AL, AM, AT, AZ, BA, BE, BG, BY, CH, CW, C...",1,251880,False,{'isrc': 'GBN9Y1100001'},{'spotify': 'https://open.spotify.com/track/3z...,https://api.spotify.com/v1/tracks/3z7dWKRsjDNM...,3z7dWKRsjDNM24ohLKZBnA,...,NaN,NaN,dany,1967,"[rock, rock, rock, rock, rock, rock, rock]",2022-12-30 08:42:31+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12425,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,193853,False,{'isrc': 'GBLTP1700005'},{'spotify': 'https://open.spotify.com/track/1V...,https://api.spotify.com/v1/tracks/1VofMhhL98pe...,1VofMhhL98pewltVGBSmCW,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:07+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12426,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,295493,False,{'isrc': 'GBLTP1700011'},{'spotify': 'https://open.spotify.com/track/0K...,https://api.spotify.com/v1/tracks/0KE7apgczHNY...,0KE7apgczHNYiXIvMUY0Fc,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:13+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12427,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,467306,False,{'isrc': 'GBLTP1700014'},{'spotify': 'https://open.spotify.com/track/6v...,https://api.spotify.com/v1/tracks/6vbRA9yAAgIX...,6vbRA9yAAgIXtDlmhyNqPq,...,NaN,N

# Logistic Matrix Factorization for Implicit Feedback Data (Logistic MF)
From [Christopher C. Johnson - Logistic Matrix Factorization for Implicit Feedback Data](https://web.stanford.edu/~rezab/nips2014workshop/submits/logmat.pdf)

## Setup the dataset

In [3]:
df_matrix_mf = df.copy()
df_matrix_mf = df_matrix_mf[df["type"].isin(["top_track"])]
df_matrix_mf["username"] = df_matrix_mf["username"].astype("category")
df_matrix_mf["id"] = df_matrix_mf["id"].astype("category")
df_matrix_mf[["username", "id", "affinity"] + spoti.NUMERICAL_FEATURES]
# df_matrix_mf["affinity"] *= 100
df_matrix_mf["affinity"]

0        1.00
1        0.98
2        0.96
3        0.94
4        0.92
         ... 
12318    0.10
12319    0.08
12320    0.06
12321    0.04
12322    0.02
Name: affinity, Length: 1200, dtype: float64

In [5]:
num_users = len(df_matrix_mf["username"].unique())
num_items = len(df_matrix_mf["id"].unique())
num_users, num_items

(8, 923)

In [6]:
matrix_mf = MatrixDataset(num_users, num_items)
matrix_mf.fill_from_df(df_matrix_mf["username"].cat.codes, df_matrix_mf["id"].cat.codes, df_matrix_mf["affinity"])
R = matrix_mf.matrix
R

array([[0.  , 0.  , 0.62, ..., 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , ..., 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , ..., 0.  , 0.  , 0.54],
       ...,
       [0.  , 0.  , 0.  , ..., 0.  , 0.  , 0.  ],
       [0.44, 0.6 , 0.  , ..., 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , ..., 0.92, 0.  , 0.  ]])

In [7]:
def convert_to_ids(values: List[str], column: str) -> List[int]:
    """
    Gets the user ids from the usernames.
    :param usernames: The usernames.
    :return: The user ids.
    """
    return df_matrix_mf[df_matrix_mf[column].isin(values)][column].cat.codes.tolist()

def retrieve_value_from_ids(ids: List[int], column: str) -> str:
    """
    Gets the value from the ids.
    :param ids: The ids.
    :return: The value.
    """
    return df_matrix_mf[df_matrix_mf[column].cat.codes.isin(ids)][column].tolist()

def get_df_rows_from_ids(ids: List[int], column: str, search_in: pd.DataFrame) -> pd.DataFrame:
    """
    Gets the dataframe rows from the ids.
    :param ids: The ids.
    :return: The dataframe rows.
    """
    return search_in[search_in[column].cat.codes.isin(ids)]

In [8]:
# Create a matrix U that contains the index that sorts the users by their affinity
I = np.argsort(matrix_mf.matrix, axis=1)
I.shape

(8, 923)

In [9]:
def compute_alpha(matrix: np.ndarray) -> np.ndarray:
    """
    Computes the alpha matrix.
    :param matrix: The matrix.
    :param k: The number of neighbors.
    :return: The alpha matrix.
    """
    alpha = len(np.where(matrix == 0)[0]) / matrix.sum()
    return alpha

alpha = compute_alpha(matrix_mf.matrix)
print(alpha)
R *= alpha
R

13.540265635507735


array([[ 0.        ,  0.        ,  8.39496469, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  7.31174344],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 5.95771688,  8.12415938,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ..., 12.45704438,
         0.        ,  0.        ]])

In [10]:
lmf = LogisticMatrixFactorization(
    factors=50,
    regularization=0.01,
    use_gpu=False,
    iterations=100,
    num_threads=1,
    random_state=42,
    dtype=np.float32,
)

R = csr_matrix(R)
lmf.fit(R, show_progress=True)

# recommend items for a user
user = "owen"
user_id = convert_to_ids([user], "username")[0]
user_items = R[user_id]
ids, scores = lmf.recommend(user_id, R[user_id], N=10, filter_already_liked_items=False)
ids = retrieve_value_from_ids(ids, "id")

df_recommended = df_matrix_mf[df_matrix_mf["id"].isin(ids)]
df_recommended[spoti.PRETTY_PRINT_FEATURES]

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [00:00<00:00, 261.82it/s]


,username,artists_names,name,release_year,popularity,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence,tempo,loudness,duration_ms,release_year,popularity
1516,owen,Hamza,Free YSL,2023,62,0.907,0.577,0.0574,0.063400,0.000010,0.1580,0.459,125.981,-8.146,186253,2023,62
1528,owen,betcover!!,海豚少年 - アルバムバージョン,2019,21,0.681,0.708,0.0912,0.117000,0.011000,0.1000,0.324,100.386,-7.375,293733,2019,21
1545,owen,Cinco,G13,2022,46,0.599,0.536,0.1120,0.418000,0.000000,0.2310,0.671,105.160,-9.528,150000,2022,46
1551,owen,Dosseh,Macabre,2023,56,0.815,0.648,0.3030,0.559000,0.000000,0.1080,0.707,140.043,-6.465,208841,2023,56
1556,owen,betcover!!,海豚少年 - アルバムバージョン,2019,21,0.681,0.708,0.0912,0.117000,0.011000,0.1000,0.324,100.386,-7.375,293733,2019,21
1559,owen,Cinco,G13,2022,46,0.599,0.536,0.1120,0.418000,0.000000,0.2310,0.671,105.160,-9.528,150000,2022,46
1561,owen,Dosseh,Macabre,2023,56,0.815,0.648,0.3030,0.559000,0.000000,0.1080,0.707,140.043,-6.465,208841,2023,56
1568,owen,Saez,J'accuse,2010,54,0.743,0.712,0.0624,0.075900,0.000000,0.0785,0.884,107.013,-5.018,268532,2010,54
1569,owen,Saez,No Place For Us,2002,26,0.533,0.662,0.0586,0.017400,0.000003,0.2180,0.727,154.493,-8.137,216053,2002,26
1570,owen,Zola,COEUR DE ICE (feat. Damso),2023,69,0.851,0.567,0.2310,0.215000,0.000000,0.0998,0.446,119.969,-8.074,192213,2023,69


_Let $l_{u,i}$ denote the event that user $u$ has chosen to interact with item $i$ (user $u$ prefers item $i$). Then, we can let the probability of this event occurring be distributed according to a logistic function parameterized by the sum of the inner product of user and item latent factor vectors and user and item biases._
$$
p(l_{ui} | x_u, y_i, \beta_i, \beta_j) = \frac{\exp(x_uy_i^T + \beta_u + \beta_i)}{1 + \exp(x_uy_i^T + \beta_u + \beta_i)}
$$



The log posterior probability of $x_u, y_i, \beta_u, \beta_i$ given the observed data $\mathbf{R}$ is given by:
$$
\log p(\mathbf{X}, \mathbf{Y}, \beta | \mathbf{R}) = \sum_{u,i} \alpha r_{ui} (x_u y_i^T + \beta_u + \beta_i) - (1 + \alpha r_{ui}) \log(1 + \exp(x_u y_i^T + \beta_u + \beta_i)) - \frac{\lambda}{2} ||x_u||^2 - \frac{\lambda}{2} ||y_i||^2
$$

In [11]:
def log_posterior(
    X: torch.Tensor,
    Y: torch.Tensor,
    beta_u: torch.Tensor,
    beta_i: torch.Tensor,
    R: torch.Tensor,
    alpha: float,
    lambd: float,
) -> torch.Tensor:
    """
    Computes the log posterior of the model, which is:
    :param X: The latent vectors of the users.
    :param Y: The latent vectors of the items.
    :param beta_u: The bias of the users.
    :param beta_i: The bias of the items.
    :param R: The matrix of interactions.
    :param alpha: The alpha parameter.
    :param lambd: The lambda parameter.
    :return: The log posterior.
    """
    # Compute the product x_u * y_i^T + beta_u + beta_i for all user-item pairs
    user_item_interactions = torch.matmul(X, Y.t()) + beta_u[:, None] + beta_i[None, :]

    # Compute the log posterior term by term
    log_posterior = torch.sum(
        alpha * R * user_item_interactions - 
        (1 + alpha * R) * torch.log1p(torch.exp(user_item_interactions))
    )

    # Subtract regularization terms
    log_posterior -= lambd / 2 * (torch.sum(X**2) + torch.sum(Y**2))

    return log_posterior

In [12]:
def mpr(I: torch.Tensor, R: torch.Tensor) -> float:
    """
    Compute the Mean Percentile Ranking (MPR) using a sorted index matrix.

    :param I: Matrix of sorted indices of items for each user.
    :param R: Rating or interaction matrix.
    :return: MPR value.
    """
    num_users, num_items = R.shape
    total_interactions = torch.sum(R)

    # Initialize MPR
    mpr = 0.0

    # Iterate over each user
    for u in range(num_users):
        # Get the indices of the items in sorted order for this user
        sorted_indices = I[u]

        # Calculate the rank for each item
        for i in range(num_items):
            item_index = sorted_indices[i]
            rank = i / num_items  # Percentile rank
            mpr += R[u, item_index] * rank

    # Normalize by the total number of interactions
    mpr /= total_interactions

    return mpr

In [13]:
def compute_predictions(
    X: torch.Tensor,
    Y: torch.Tensor,
    beta_u: torch.Tensor,
    beta_i: torch.Tensor,
) -> torch.Tensor:
    """
    Computes the predictions of the model.
    :param X: The latent vectors of the users.
    :param Y: The latent vectors of the items.
    :param beta_u: The bias of the users.
    :param beta_i: The bias of the items.
    :param R: The matrix of interactions.
    :return: The predictions.
    """
    return torch.matmul(X, Y.t()) + beta_u[:, None] + beta_i

In [14]:
def train(
    R: np.ndarray,
    num_users: int,
    num_items: int,
    num_latent: int,
    lambd: float,
    alpha: float,
    learning_rate: float,
    epochs: int,
    mprs: List[float] = [],
    losses: List[float] = [],
):
    """
    Trains the model.
    :param num_latent: The number of latent factors.
    :param lambd: The lambda parameter.
    :param alpha: The alpha parameter.
    :param learning_rate: The learning rate.
    :param epochs: The number of epochs.
    """
    mprs = []  # Record MPRs for each epoch
    losses = []  # Record losses for each epoch

    # Initialize user and item latent factor matrices and bias vectors
    R = torch.tensor(R, dtype=torch.float64)
    X = torch.randn(num_users, num_latent, requires_grad=False, dtype=torch.float64)
    Y = torch.randn(num_items, num_latent, requires_grad=False, dtype=torch.float64)
    beta_u = torch.randn(num_users, requires_grad=False, dtype=torch.float64)
    beta_i = torch.randn(num_items, requires_grad=False, dtype=torch.float64)

    # Initialize optimizer
    optimizer = optim.Adagrad([X, Y, beta_u, beta_i], lr=learning_rate)

    # Training loop
    for epoch in range(epochs):
        # Fix X and B and take a step toward Y and B
        X.requires_grad = False
        Y.requires_grad = True
        beta_u.requires_grad = False
        beta_i.requires_grad = True
        optimizer.zero_grad()
        loss = -log_posterior(X, Y, beta_u, beta_i, R, alpha, lambd)
        loss.backward()
        optimizer.step()

        # Fix Y and B and take a step toward X and B
        X.requires_grad = True
        Y.requires_grad = False
        beta_u.requires_grad = True
        beta_i.requires_grad = False
        optimizer.zero_grad()
        loss = -log_posterior(X, Y, beta_u, beta_i, R, alpha, lambd)
        loss.backward()
        optimizer.step()

        # Make predictions
        predictions = compute_predictions(X, Y, beta_u, beta_i)

        # Compute MPR
        I = torch.argsort(predictions, descending=True, dim=1)
        mpr_value = mpr(I, R)

        # Record MPR
        mprs.append(mpr_value)

        # Record loss
        losses.append(loss.item())

        # Print progress
        print(f"Epoch {epoch + 1} - Loss: {loss.item():.4f} - MPR: {mpr_value:.4f}")

    return X, Y, beta_u, beta_i, mprs, losses

In [15]:
def train_with_gradients(
    R: np.ndarray,
    num_users: int,
    num_items: int,
    num_latent: int,
    lambd: float,
    alpha: float,
    learning_rate: float,
    epochs: int,
    mprs: List[float] = [],
    losses: List[float] = [],
):
    """
    Trains the model.
    :param num_latent: The number of latent factors.
    :param lambd: The lambda parameter.
    :param alpha: The alpha parameter.
    :param learning_rate: The learning rate.
    :param epochs: The number of epochs.
    """

    # Initialize user and item latent factor matrices and bias vectors
    R = torch.tensor(R, dtype=torch.float64)
    X = torch.randn(num_users, num_latent, requires_grad=False, dtype=torch.float64)
    Y = torch.randn(num_items, num_latent, requires_grad=False, dtype=torch.float64)
    beta_u = torch.randn(num_users, requires_grad=False, dtype=torch.float64)
    beta_i = torch.randn(num_items, requires_grad=False, dtype=torch.float64)

    grad_accumulator_X = torch.zeros_like(X)
    grad_accumulator_Y = torch.zeros_like(Y)

    for epoch in range(epochs):
        # Fix X and B and take a step toward Y and B
        for i in range(num_items):
            term1 = alpha * R[:, i][:, None]

            exp_term = torch.exp(torch.matmul(Y[i], X.t()) + beta_u + beta_i[i])
            exp_term = exp_term[:, None]

            gradients_Y = torch.sum(term1 * X - X * (1 + term1) * exp_term / (1 + exp_term), dim=0) - lambd * Y[i]
            gradients_beta_i = torch.sum(term1 - (1 + term1) * exp_term / (1 + exp_term), dim=0).squeeze()

            grad_accumulator_Y[i] += gradients_Y ** 2
            Y[i] += learning_rate * gradients_Y / torch.sqrt(grad_accumulator_Y[i])
            beta_i[i] += learning_rate * gradients_beta_i
        
        # Fix Y and B and take a step toward X and B
        for u in range(num_users):
            term1 = alpha * R[u, :][:, None]

            exp_term = torch.exp(torch.matmul(X[u], Y.t()) + beta_u[u] + beta_i)
            exp_term = exp_term[:, None]

            gradients_X = torch.sum(term1 * Y - Y * (1 + term1) * exp_term / (1 + exp_term), dim=0) - lambd * X[u]
            gradients_beta_u = torch.sum(term1 - (1 + term1) * exp_term / (1 + exp_term), dim=0).squeeze()

            grad_accumulator_X[u] += gradients_X ** 2
            X[u] += learning_rate * gradients_X / torch.sqrt(grad_accumulator_X[u])
            beta_u[u] += learning_rate * gradients_beta_u

        # Make predictions
        predictions = compute_predictions(X, Y, beta_u, beta_i)

        # Compute MPR
        I = torch.argsort(predictions, descending=True, dim=1)
        mpr_value = mpr(I, R)

        # Record MPR
        mprs.append(mpr_value)

        # Record loss
        loss = -log_posterior(X, Y, beta_u, beta_i, R, alpha, lambd)
        losses.append(loss.item())

        # Print progress
        print(f"Epoch {epoch + 1} - Loss: {loss.item():.4f} - MPR: {mpr_value:.4f}")

    return X, Y, beta_u, beta_i, mprs, losses

In [16]:
num_latent = 100
lambd = 0.01
mprs = []
losses = []
X, Y, beta_u, beta_i, mprs, losses = train(
    R=R,
    num_latent=num_latent,
    num_users=num_users,
    num_items=num_items,
    lambd=lambd,
    alpha=alpha,
    learning_rate=0.1,
    epochs=500,
    mprs=mprs,
    losses=losses,
)

TypeError: sparse array length is ambiguous; use getnnz() or shape[0]

In [ ]:
px.line(y=mprs, title="MPR over epochs").show()
px.line(y=losses, title="Loss over epochs").show()

In [ ]:
user_latent = X.detach().numpy()
item_latent = Y.detach().numpy()

In [ ]:
if num_latent <= 3:
    # Add the latent vectors to the dataframe
    df_matrix_mf["user_latent"] = df_matrix_mf["username"].cat.codes.apply(
        lambda x: user_latent[x]
    )
    df_matrix_mf["item_latent"] = df_matrix_mf["id"].cat.codes.apply(
        lambda x: item_latent[x]
    )

    # Add the biases to the dataframe
    df_matrix_mf["user_bias"] = df_matrix_mf["username"].cat.codes.apply(
        lambda x: beta_u[x].item()
    )
    df_matrix_mf["item_bias"] = df_matrix_mf["id"].cat.codes.apply(
        lambda x: beta_i[x].item()
    )

    for i in range(num_latent):
        df_matrix_mf[f"user_latent_{i}"] = df_matrix_mf["user_latent"].apply(lambda x: x[i])
        df_matrix_mf[f"latent_{i}"] = df_matrix_mf["item_latent"].apply(lambda x: x[i])

    fig = plotting.plot_latent_space(
        df_matrix_mf,
        color=df_matrix_mf["username"],
        text=df_matrix_mf["username"],
        title="User latent space",
        latent_columns=["latent_0", "latent_1", "latent_2"],
    )
    fig.show()

    df_user_latent = df_matrix_mf.drop_duplicates(subset=["username"])
    fig = plotting.plot_latent_space(
        df_user_latent,
        color=df_user_latent["username"],
        text=df_user_latent["username"],
        title="User latent space",
        latent_columns=["user_latent_0", "user_latent_1", "user_latent_2"],
    )
    fig.show()

In [ ]:
# Perform recommendations

# Get the latent vectors of the users
user_latent = X.detach().numpy()

# Get the latent vectors of the items
item_latent = Y.detach().numpy()

# Get the biases of the users
user_bias = beta_u.detach().numpy()

# Get the biases of the items
item_bias = beta_i.detach().numpy()

C = cosine_similarity(user_latent, item_latent)

# Get the top 10 recommendations for each user
top_k = 120
top_k_recommendations = np.argsort(C, axis=1)[:, -top_k:]

print(top_k_recommendations)

# Show the recommendations for a user
user_index = 7
user_id = df_matrix_mf["username"].cat.categories[user_index]
print(f"Recommendations for user {user_id}:")
for item_index in top_k_recommendations[user_index]:
    item = df_matrix_mf[df_matrix_mf["id"].cat.codes == item_index].iloc[0]
    if item["username"] != user_id:
        print(f"  - {item['name']} by {item['artists_names']} (from {item['username']})")

[[858 538 299 679 395 805  28 818 463 822 675 627 292 766 540   7 750 276
  433 741 428 853 224 382 873 838 715   8 185 632 234 319 196 133 103 246
  539 648 256 622 231  74 914 908 598 106 579 756 879  30 208  78 571  59
  563 476 776 580 800 553 439 529 724 156 568 237  18 900 182 295 840 320
  729 898 874 904 127  14  24 250 676 561 610  25 509 379 249 816 260 348
  303 678 716  20 413 566 662 570 122  90 105 641 259 408  52 609 410 639
  799 634 398 705 346 700 383 139   2 728 170 859]
 [871 804 870 358 540 105  52 489 139 750 403 835 664 770 195 257 252 868
  238 740  38 651 790 882 354 306 160 391 136 667  39 478 326 282 538  85
   92 701 263 158 565 525 681 145 520 481  87 687 897 880 523  43 820 455
  215 559  55  33 753 204 663  23 611 902 230  64 906 337 460 333 217 206
  658 679 125 577 335 120 482 229 693 629 818 603  40 387 186 299 650 277
  837 895 256 109 372 313 775 285 115 624 722 849 154 556 888 461  68 727
  475 597 435 423 627 340 814 600 769 331 151 499]
 [275  49 

Save the model

In [ ]:
# save the model
# torch.save(X, "models/X.pt")
# torch.save(Y, "models/Y.pt")
# torch.save(beta_u, "models/beta_u.pt")
# torch.save(beta_i, "models/beta_i.pt")
# torch.save(alpha, "models/alpha.pt")
# torch.save(lambd, "models/lambd.pt")